In [22]:
from pathlib import Path
import io
import pandas as pd
from credit_card_fraud_detection.paths import ROOT, RAW
from itertools import islice

In [ ]:
def require(*paths: Path) -> None:
    missing = [str(p) for p in paths if not p.is_file()]
    if missing:
        raise FileNotFoundError("missing inputs:\n  " + "\n  ".join(missing))


def check_input(path: Path) -> Path:
    if not path.is_file():
        raise FileNotFoundError(f"Input not found: {path.resolve()}")
    if path.stat().st_size == 0:
        raise ValueError(f"input is empty: {path.resolve()}")
    return path


# check first few raw lines
def peek(path: Path, n: int=5, encoding: str = "utf-8") -> None:
    with path.open(encoding=encoding, errors="replace") as fh:
        for line in islice(fh, n):
            print(repr(line))

In [14]:
print(ROOT)
print(sorted(p.name for p in RAW.glob("*")))

C:\Users\shank\OneDrive\Documents\Data analysis\credit_card_fraud_detection
['.gitkeep', 'sample_submission.csv', 'test.csv', 'train.csv']


In [15]:
train = RAW / "train.csv"
test = RAW / "test.csv"

In [16]:
check_input(train)

WindowsPath('C:/Users/shank/OneDrive/Documents/Data analysis/credit_card_fraud_detection/data/raw/train.csv')

In [21]:
# check first few raw lines
peek(train)

'id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,time_of_day_hour,day_of_week,merchant_risk_score,prior_disputes,merchant_category,card_type,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud\n'
'0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,17,3,29.2,0,Groceries,Mastercard,OTP,POS,iPhone,False,False,False,False,False,False,0.0\n'
'1,24.83,1.54,3,13.06,38,28,3233.0,0,33.4,23,5,29.9,1,Online Retail,Visa,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0\n'
'2,6.36,1.2,4,8.17,50,55,1046.88,0,35.8,15,0,24.1,0,Utilities,Visa,OTP,Online,Tablet,False,False,True,False,False,False,0.0\n'
'3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,7,3,41.9,0,Groceries,Mastercard,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0\n'


In [26]:
# know total size of csv , under 500mb = read normally, over 500mb = read a sample with nrows to design dtypes, then read only the needed columns or process in chunks

#if you need to make dummy csv use the commented line below
#csv = io.StringIO("id, amount\n1,10\n2,20\n3,30\n4,40\n")

#Initialize a running total for our target column
#total=0

# use this method to Read the CSV in small batches (chunks) instead of loading the entire file into RAM at once.
#'usecols' loads only the single needed column, ignoring the rest to save memory and speed up processing.
'''for chunk in pd.read_csv(train, chunksize=1000, usecols=["amount_usd"]):
    total += chunk["amount_usd"].sum()
print(total)'''

'for chunk in pd.read_csv(train, chunksize=1000, usecols=["amount_usd"]):\n    total += chunk["amount_usd"].sum()\nprint(total)'

In [ ]:
# set encoding
 
train_df = pd.read_csv(train, encoding="utf-8")
train_df